In [2]:
import heapq

# 1. CẤU HÌNH BÀI TOÁN
# Ô số 0 đại diện cho ô trống
START_STATE = ((1, 2, 3), (4, 0, 6), (7, 5, 8))
GOAL_STATE  = ((1, 2, 3), (4, 5, 6), (7, 8, 0))

# Các hướng di chuyển của ô trống (Thứ tự ưu tiên chuẩn: Trái -> Phải -> Lên -> Xuống)
ACTIONS = [
    ('L', (0, -1), "SANG TRÁI"),
    ('R', (0, 1), "SANG PHẢI"),
    ('U', (-1, 0), "LÊN"),
    ('D', (1, 0), "XUỐNG")
]

def find_zero(state):
    for i in range(3):
        for j in range(3):
            if state[i][j] == 0:
                return i, j
    return -1, -1

def swap(state, x1, y1, x2, y2):
    state = [list(row) for row in state]
    state[x1][y1], state[x2][y2] = state[x2][y2], state[x1][y1]
    return tuple(tuple(row) for row in state)

# Hàm Heuristic: Tính tổng khoảng cách Manhattan từ trạng thái hiện tại đến đích
def manhattan_distance(current, goal):
    goal_pos = {}
    for r in range(3):
        for c in range(3):
            goal_pos[goal[r][c]] = (r, c)

    distance = 0
    for r in range(3):
        for c in range(3):
            val = current[r][c]
            if val != 0:  # Không tính khoảng cách cho ô trống
                target_r, target_c = goal_pos[val]
                distance += abs(r - target_r) + abs(c - target_c)
    return distance

# THUẬT TOÁN A* (A-Star Search)
def run_astar(start, goal):
    step_count = 0
    frontier = []

    # Khởi tạo h(n) và f(n) cho nút gốc
    h_start = manhattan_distance(start, goal)
    # Lưu vào Heap định dạng: (f_cost, step_count, g_cost, current_state, path)
    heapq.heappush(frontier, (h_start, step_count, 0, start, []))

    explored = set()
    frontier_costs = {start: 0} # Lưu trữ chi phí g tốt nhất từng đi qua đến nút này

    while frontier:
        f_cost, _, g_cost, current, path = heapq.heappop(frontier)

        if current in explored:
            continue
        explored.add(current)

        # Kiểm tra nếu chạm đích
        if current == goal:
            return path, len(explored)

        r, c = find_zero(current)
        for move, (dx, dy), move_name in ACTIONS:
            new_r, new_c = r + dx, c + dy
            if 0 <= new_r < 3 and 0 <= new_c < 3:
                next_state = swap(current, r, c, new_r, new_c)

                # Chi phí bước đi thông thường trong 8-puzzle chuẩn là 1 bước
                new_g = g_cost + 1

                if next_state in explored:
                    continue

                # Nếu tìm thấy đường đi ngắn hơn (chi phí g nhỏ hơn) hoặc nút chưa từng thăm
                if next_state not in frontier_costs or new_g < frontier_costs[next_state]:
                    frontier_costs[next_state] = new_g
                    new_h = manhattan_distance(next_state, goal)
                    new_f = new_g + new_h

                    step_count += 1
                    heapq.heappush(frontier, (new_f, step_count, new_g, next_state, path + [move]))

    return None, len(explored)

if __name__ == "__main__":
    print("=== ĐANG CHẠY THUẬT TOÁN A* ===")
    path, nodes_visited = run_astar(START_STATE, GOAL_STATE)

    if path is not None:
        print(f"THÀNH CÔNG tìm thấy đường đi!")
        print(f"-> Tổng số Node đã duyệt qua: {nodes_visited}")
        print(f"-> Số bước di chuyển: {len(path)}")
        print(f"-> Chi tiết lộ trình: {' -> '.join(path)}")
    else:
        print("Không tìm thấy đường đi giải bài toán.")

=== ĐANG CHẠY THUẬT TOÁN A* ===
THÀNH CÔNG tìm thấy đường đi!
-> Tổng số Node đã duyệt qua: 3
-> Số bước di chuyển: 2
-> Chi tiết lộ trình: D -> R


2. Greedy Best-First Search

In [3]:
import heapq

# 1. CẤU HÌNH BÀI TOÁN
START_STATE = ((1, 2, 3), (4, 0, 6), (7, 5, 8))
GOAL_STATE  = ((1, 2, 3), (4, 5, 6), (7, 8, 0))

ACTIONS = [
    ('L', (0, -1), "SANG TRÁI"),
    ('R', (0, 1), "SANG PHẢI"),
    ('U', (-1, 0), "LÊN"),
    ('D', (1, 0), "XUỐNG")
]

def find_zero(state):
    for i in range(3):
        for j in range(3):
            if state[i][j] == 0:
                return i, j
    return -1, -1

def swap(state, x1, y1, x2, y2):
    state = [list(row) for row in state]
    state[x1][y1], state[x2][y2] = state[x2][y2], state[x1][y1]
    return tuple(tuple(row) for row in state)

def manhattan_distance(current, goal):
    goal_pos = {}
    for r in range(3):
        for c in range(3):
            goal_pos[goal[r][c]] = (r, c)

    distance = 0
    for r in range(3):
        for c in range(3):
            val = current[r][c]
            if val != 0:
                target_r, target_c = goal_pos[val]
                distance += abs(r - target_r) + abs(c - target_c)
    return distance

# THUẬT TOÁN THAM LAM (Greedy Best-First Search)
def run_greedy(start, goal):
    step_count = 0
    frontier = []

    # Greedy chỉ quan tâm và đẩy vào Heap theo giá trị h(n)
    h_start = manhattan_distance(start, goal)
    heapq.heappush(frontier, (h_start, step_count, start, []))

    explored = set()

    while frontier:
        h_cost, _, current, path = heapq.heappop(frontier)

        if current in explored:
            continue
        explored.add(current)

        if current == goal:
            return path, len(explored)

        r, c = find_zero(current)
        for move, (dx, dy), move_name in ACTIONS:
            new_r, new_c = r + dx, c + dy
            if 0 <= new_r < 3 and 0 <= new_c < 3:
                next_state = swap(current, r, c, new_r, new_c)

                if next_state not in explored:
                    new_h = manhattan_distance(next_state, goal)
                    step_count += 1
                    # Chỉ sắp xếp hàng đợi theo độ lớn của new_h
                    heapq.heappush(frontier, (new_h, step_count, next_state, path + [move]))

    return None, len(explored)

if __name__ == "__main__":
    print("=== ĐANG CHẠY THUẬT TOÁN GREEDY SEARCH ===")
    path, nodes_visited = run_greedy(START_STATE, GOAL_STATE)

    if path is not None:
        print(f"THÀNH CÔNG tìm thấy đường đi!")
        print(f"-> Tổng số Node đã duyệt qua: {nodes_visited}")
        print(f"-> Số bước di chuyển: {len(path)}")
        print(f"-> Chi tiết lộ trình: {' -> '.join(path)}")
    else:
        print("Không tìm thấy đường đi giải bài toán.")

=== ĐANG CHẠY THUẬT TOÁN GREEDY SEARCH ===
THÀNH CÔNG tìm thấy đường đi!
-> Tổng số Node đã duyệt qua: 3
-> Số bước di chuyển: 2
-> Chi tiết lộ trình: D -> R
